# Student Performance Exploratory Data Analysis

This notebook walks through real-world student performance analysis with a messy dataset. Students will: 
- use Pandas for data loading and transformation
- clean messy values and handle missing/duplicate records
- explore grouping and aggregation
- create analytical features
- visualize distributions and relationships
- interpret correlation
- export cleaned data and reports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
data_path = os.path.join("data", "student_performance.csv")

In [ ]:
raw_df = pd.read_csv(data_path, dtype=str)
raw_df.columns = raw_df.columns.str.strip()
print("Raw dataset shape:", raw_df.shape)
raw_df.head(10)

In [ ]:
def parse_numeric(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.str.replace(r"%", "", regex=True)
    cleaned = cleaned.str.replace(r"[^0-9.\-]+", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

def normalize_gender(series: pd.Series) -> pd.Series:
    normalized = series.astype(str).str.strip().str.lower()
    normalized = normalized.replace({
        "m": "Male",
        "male": "Male",
        "f": "Female",
        "female": "Female",
        "n/a": "Unknown",
        "na": "Unknown",
        "": "Unknown",
    })
    normalized = normalized.where(normalized.isin(["Male", "Female"]), "Unknown")
    return normalized

def clean_student_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Student ID"] = df["Student ID"].astype(str).str.strip()
    df["Name"] = df["Name"].astype(str).str.strip()
    df["Gender"] = normalize_gender(df["Gender"])
    df["Attendance"] = parse_numeric(df["Attendance"])
    for col in ["Math", "Science", "English"]:
        df[col] = parse_numeric(df[col])
    df["GPA"] = parse_numeric(df["GPA"])
    df["Semester"] = df["Semester"].astype(str).str.strip()
    for col in ["Attendance", "Math", "Science", "English"]:
        df[col] = df[col].fillna(df[col].median())
    missing_gpa = df["GPA"].isna()
    if missing_gpa.any():
        gpa_from_marks = df.loc[missing_gpa, ["Math", "Science", "English"]].sum(axis=1).div(300).mul(4).round(2).clip(upper=4.0)
        df.loc[missing_gpa, "GPA"] = gpa_from_marks
    df["GPA"] = df["GPA"].fillna(df["GPA"].median())
    df.loc[:, ["Math", "Science", "English"]] = df[["Math", "Science", "English"]].clip(lower=0, upper=100)
    df["GPA"] = df["GPA"].clip(lower=0.0, upper=4.0)
    df = df.drop_duplicates(subset=["Student ID"], keep="first")
    return df

clean_df = clean_student_data(raw_df)
print("Cleaned dataset shape:", clean_df.shape)
clean_df.head(10)

In [ ]:
quality_summary = pd.DataFrame({
    "metric": ["Missing Attendance", "Missing Math", "Missing Science", "Missing English", "Missing GPA", "Unknown Gender", "Duplicate Student ID"],
    "raw_count": [
        raw_df["Attendance"].isna().sum(),
        raw_df["Math"].isna().sum(),
        raw_df["Science"].isna().sum(),
        raw_df["English"].isna().sum(),
        raw_df["GPA"].isna().sum(),
        raw_df["Gender"].astype(str).str.strip().isin(["", "N/A", "na"]).sum(),
        raw_df["Student ID"].duplicated().sum(),
    ],
    "cleaned_count": [
        clean_df["Attendance"].isna().sum(),
        clean_df["Math"].isna().sum(),
        clean_df["Science"].isna().sum(),
        clean_df["English"].isna().sum(),
        clean_df["GPA"].isna().sum(),
        (clean_df["Gender"] == "Unknown").sum(),
        clean_df["Student ID"].duplicated().sum(),
    ],
})
quality_summary

## Feature Engineering and Aggregation

Create new analytical columns and compute group-level summaries.

In [ ]:
clean_df = clean_df.copy()
clean_df["TotalMarks"] = clean_df[["Math", "Science", "English"]].sum(axis=1)
clean_df["AverageMarks"] = clean_df[["Math", "Science", "English"]].mean(axis=1).round(1)
clean_df["Grade"] = pd.cut(clean_df["GPA"], bins=[0.0, 2.0, 2.5, 3.0, 3.5, 4.0], labels=["F", "D", "C", "B", "A"], include_lowest=True)
clean_df["Status"] = clean_df["GPA"].apply(lambda x: "Pass" if x >= 2.0 else "Fail")

subject_avg = clean_df[["Math", "Science", "English"]].mean().round(2)
attendance_by_gender = clean_df.groupby("Gender")["Attendance"].mean().round(2)
gpa_by_semester = clean_df.groupby("Semester")["GPA"].mean().round(2)
grade_counts = clean_df["Grade"].value_counts().sort_index()

subject_avg, attendance_by_gender, gpa_by_semester, grade_counts

## Visualizations

Plot distributions, relationships, and correlations to interpret the dataset visually.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(clean_df["GPA"], bins=20, kde=True, color="teal")
plt.title("GPA Distribution")
plt.xlabel("GPA")
plt.savefig(os.path.join(output_dir, "gpa_distribution.png"))
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(x="Gender", y="Attendance", data=clean_df, palette=["#4c72b0", "#dd8452"])
plt.title("Attendance by Gender")
plt.savefig(os.path.join(output_dir, "attendance_by_gender.png"))
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=clean_df, x="Attendance", y="AverageMarks", hue="Gender", alpha=0.7)
plt.title("Attendance vs Average Marks")
plt.savefig(os.path.join(output_dir, "attendance_vs_average_marks.png"))
plt.show()

corr = clean_df[["Attendance", "Math", "Science", "English", "GPA", "TotalMarks", "AverageMarks"]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.savefig(os.path.join(output_dir, "correlation_matrix.png"))
plt.show()

## Export Cleaned Data and Reports

Save the cleaned dataset and a summary report for future analysis or presentation.

In [ ]:
cleaned_path = os.path.join(output_dir, "student_performance_cleaned.csv")
clean_df.to_csv(cleaned_path, index=False)
report_path = os.path.join(output_dir, "eda_summary.txt")
with open(report_path, "w", encoding="utf-8") as report_file:
    report_file.write("Student Performance EDA Summary\n")
    report_file.write("============================\n")
    report_file.write(f"Raw rows: {len(raw_df)}\n")
    report_file.write(f"Cleaned rows: {len(clean_df)}\n")
    report_file.write("\nSubject averages:\n")
    report_file.write(subject_avg.to_string())
    report_file.write("\n\nAttendance by gender:\n")
    report_file.write(attendance_by_gender.to_string())
    report_file.write("\n\nSemester GPA averages:\n")
    report_file.write(gpa_by_semester.to_string())
    report_file.write("\n\nGrade counts:\n")
    report_file.write(grade_counts.to_string())

## Interpretation and Insights

Use the visualizations and aggregated results to answer questions such as:
- Which subjects have the highest and lowest average scores?
- Does gender relate to attendance or GPA in this dataset?
- How strongly are attendance and marks correlated?
- Which semesters show stronger academic performance?
- What data issues were fixed during cleaning?